In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class ModelArgs:
    # 基础维度
    hidden_size: int = 2048
    ffn_hidden_size: int = 4096 # 用于非 MoE 层或 Dense 层，MoE 层用下面的参数
    
    # MoE 核心参数
    num_experts: int = 64
    moe_ffn_hidden_size: int = 768  # 路由专家维度
    moe_router_topk: int = 4
    moe_shared_expert_intermediate_size: int = 768 # 共享专家维度
    
    # Router 机制参数
    moe_router_score_function: str = 'sigmoid' # 关键点：使用 Sigmoid
    moe_router_enable_expert_bias: bool = True
    moe_router_bias_update_rate: float = 0.001
    moe_router_topk_scaling_factor: float = 1.0 # 有些实现中会用
    
    # 激活函数与初始化
    swiglu: bool = True
    disable_bias_linear: bool = True # FFN 内部线性层无 Bias
    init_method_std: float = 0.02

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SwiGLUBlock(nn.Module):
    def __init__(self, args: ModelArgs, hidden_dim: int):
        super().__init__()
        # 1. 门控投影 (Gate Projection): 负责决定哪些信息通过
        self.gate_proj = nn.Linear(args.hidden_size, hidden_dim, bias=False)
        
        # 2. 上行投影 (Up Projection): 负责提取特征信息
        self.up_proj = nn.Linear(args.hidden_size, hidden_dim, bias=False)
        
        # 3. 下行投影 (Down Projection): 将维度映射回原始大小
        self.down_proj = nn.Linear(hidden_dim, args.hidden_size, bias=False)

    def forward(self, x: torch.Tensor):
        # 步骤 A: 计算门控分支
        # gate_output 形状: (batch, seq_len, hidden_dim)
        gate_output = self.gate_proj(x)
        
        # 步骤 B: 对门控分支应用 SiLU 激活函数 (也叫 Swish)
        # swish(x) = x * sigmoid(x)
        activated_gate = F.silu(gate_output)
        
        # 步骤 C: 计算特征分支 (Up Projection)
        # up_output 形状: (batch, seq_len, hidden_dim)
        up_output = self.up_proj(x)
        
        # 步骤 D: 逐元素相乘 (Element-wise Multiplication)
        # 这是 SwiGLU 的核心：用激活后的门控信号去“乘”特征信号
        intermediate = activated_gate * up_output
        
        # 步骤 E: 最终线性投影
        # 将 hidden_dim 映射回 args.hidden_size
        output = self.down_proj(intermediate)
        
        return output

In [ ]:
class DeepSeekRouter(nn.Module):
    """
    实现 DeepSeek V3 的 Sigmoid 路由机制。
    对应参数: 
    --moe-router-score-function sigmoid
    --moe-router-enable-expert-bias
    --moe-router-topk
    """
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.topk = args.moe_router_topk
        
        # 路由器线性层
        # 参数: --moe-router-enable-expert-bias
        self.gate = nn.Linear(args.hidden_size, args.num_experts, bias=True)
        
        # 负载均衡用的 bias (DeepSeek V3 Aux-free Load Balancing)
        # 这是一个在训练中动态更新的 buffer，不参与梯度下降，但参与 forward 计算
        self.register_buffer('expert_bias', torch.zeros(args.num_experts))

    def forward(self, x):
        # x: (Batch, Seq, Dim)
        input_shape = x.shape
        x_flat = x.view(-1, input_shape[-1])
        
        # 1. 计算 Logits
        # DeepSeek V3 建议对 router 输入做 normalize
        router_logits = self.gate(F.normalize(x_flat, dim=-1))
        
        # 2. 加上动态负载平衡 Bias (仅在训练逻辑中会更新这个 bias)
        # 对应参数: --moe-router-bias-update-rate (逻辑需在 loss 计算后 update)
        if self.training:
            router_logits = router_logits + self.expert_bias
            
        # 3. 计算分数 (Sigmoid vs Softmax)
        # 对应参数: --moe-router-score-function sigmoid
        if self.args.moe_router_score_function == 'sigmoid':
            scores = torch.sigmoid(router_logits)
        else:
            scores = F.softmax(router_logits, dim=-1)
            
        # 4. Top-K 选择
        # topk_weights: (Batch*Seq, k)
        # topk_indices: (Batch*Seq, k)
        topk_weights, topk_indices = torch.topk(scores, self.topk, dim=-1)
        
        # 5. 归一化选中的权重 (Renormalize)
        # 因为 Sigmoid 输出不是概率分布(和不为1)，选出 TopK 后需要重新归一化
        topk_weights = topk_weights / topk_weights.sum(dim=-1, keepdim=True)
        
        # 6. 可选：Scaling Factor (防止数值过小)
        # 对应参数: --moe-router-topk-scaling-factor
        if hasattr(self.args, 'moe_router_topk_scaling_factor'):
            topk_weights = topk_weights * self.args.moe_router_topk_scaling_factor

        return topk_weights, topk_indices, x_flat, input_shape

class DeepSeekV3MoELayer(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        
        # --- 1. Shared Expert (共享专家) ---
        # 对应参数: --moe-shared-expert-intermediate-size
        # 这是一个 Always-On 的独立路径
        self.shared_expert = SwiGLUBlock(args, args.moe_shared_expert_intermediate_size)
        
        # --- 2. Routed Experts (路由专家) ---
        # 对应参数: --num-experts, --moe-ffn-hidden-size
        self.routed_experts = nn.ModuleList([
            SwiGLUBlock(args, args.moe_ffn_hidden_size)
            for _ in range(args.num_experts)
        ])
        
        # --- 3. Router ---
        self.router = DeepSeekRouter(args)

    def forward(self, x):
        # x: (Batch, Seq, Dim)
        
        # === Part A: 共享专家路径 ===
        shared_output = self.shared_expert(x)
        
        # === Part B: 路由专家路径 ===
        weights, indices, x_flat, original_shape = self.router(x)
        
        # 初始化路由输出
        # (Batch*Seq, Dim)
        routed_output = torch.zeros_like(x_flat)
        
        # 简单的循环实现 (生产环境应用 Triton/CUDA 优化)
        # 遍历选中的 TopK
        for k in range(self.args.moe_router_topk):
            # 获取第 k 个选择的 expert index 和 weight
            k_indices = indices[:, k] # (B*S,)
            k_weights = weights[:, k] # (B*S,)
            
            # 遍历所有专家 (逻辑上可以优化为只遍历被选中的专家)
            for expert_idx in range(self.args.num_experts):
                # 找出哪些 token 选中了当前 expert_idx
                mask = (k_indices == expert_idx)
                
                if mask.any():
                    # 选出对应的 token
                    selected_tokens = x_flat[mask]
                    
                    # 专家计算
                    expert_out = self.routed_experts[expert_idx](selected_tokens)
                    
                    # 加权累加
                    # weight 需要 unsqueeze 以匹配维度 (N, 1) * (N, D)
                    w = k_weights[mask].unsqueeze(-1)
                    routed_output[mask] += expert_out * w
        
        # 恢复形状 (Batch, Seq, Dim)
        routed_output = routed_output.view(original_shape)
        
        # === Part C: 最终融合 ===
        # DeepSeek 方式: 直接相加
        final_output = shared_output + routed_output
        
        return final_output

In [ ]:
# 使用提供的 MODEL_ARGS 初始化配置
args = ModelArgs(
    hidden_size=2048,
    num_experts=64,
    moe_ffn_hidden_size=768,       # 细粒度专家
    moe_shared_expert_intermediate_size=768, # 共享专家
    moe_router_topk=4,
    moe_router_score_function='sigmoid',
    moe_router_enable_expert_bias=True,
    moe_router_bias_update_rate=0.001
)

print(f"DeepSeek V3 Config: {args}")

model = DeepSeekV3MoELayer(args)

# 模拟输入 (Batch=2, Seq=128, Dim=2048)
x = torch.randn(2, 128, 2048)

output = model(x)

print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")

# 验证 Sigmoid Router
print("\nCheck Router Logic:")
# 提取 router 内部的 sigmoid 行为
logits = model.router.gate(F.normalize(x.view(-1, 2048), dim=-1))
scores = torch.sigmoid(logits)
print(f"Max Score (Sigmoid): {scores.max().item():.4f} (Should be <= 1.0)")
print(f"Min Score (Sigmoid): {scores.min().item():.4f} (Should be >= 0.0)")

DeepSeek V3 Config: ModelArgs(hidden_size=2048, ffn_hidden_size=4096, num_experts=64, moe_ffn_hidden_size=768, moe_router_topk=4, moe_shared_expert_intermediate_size=768, moe_router_score_function='sigmoid', moe_router_enable_expert_bias=True, moe_router_bias_update_rate=0.001, moe_router_topk_scaling_factor=1.0, swiglu=True, disable_bias_linear=True, init_method_std=0.02)

Input shape: torch.Size([2, 128, 2048])
Output shape: torch.Size([2, 128, 2048])

Check Router Logic:
Max Score (Sigmoid): 0.5165 (Should be <= 1.0)
Min Score (Sigmoid): 0.4861 (Should be >= 0.0)


: 